# 📄 04 — Automated Report Generation

This notebook demonstrates how to build comprehensive
analysis reports programmatically.

**Output formats:**
- HTML (always available)
- PDF (requires `weasyprint`)

---

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Non-interactive for report generation
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from survey_toolkit import (
    SurveyLoader,
    SurveyCleaner,
    SurveyEDA,
    SurveyStats,
    SurveyClassifier,
    SurveySegmentation,
    ReportGenerator,
    generate_sample_survey,
    detect_column_types,
    compute_scale_scores,
    validate_survey_data,
)

print("✅ Setup complete")

## 2. Run Full Analysis Pipeline

First, we'll run the complete analysis to generate all the
results we need for the report.

### 2.1 Generate & Load Data

In [ ]:
df = generate_sample_survey(
    n_respondents=500,
    n_likert_items=10,
    n_demographic_cols=4,
    include_open_ended=True,
    random_state=42,
    save_path="../data/sample_survey.csv",
)

types = detect_column_types(df)
likert_cols = types["likert"]
demo_cols = types["categorical"]

validation = validate_survey_data(df)
print(f"✅ Data loaded: {df.shape}")
print(f"✅ Valid: {validation['valid']}")

### 2.2 Clean Data

In [ ]:
cleaner = SurveyCleaner(df)
clean_df = (
    cleaner
    .remove_speeders("duration_seconds", min_seconds=60)
    .remove_straightliners(likert_cols, threshold=0.95)
    .handle_missing(strategy="median", columns=likert_cols)
    .get_clean_data()
)
cleaning_log = cleaner.get_log()

print(f"✅ Cleaned: {df.shape} → {clean_df.shape}")
print(f"   Steps: {len(cleaning_log)}")

### 2.3 EDA & Figures

In [ ]:
eda = SurveyEDA(clean_df, output_dir="../outputs/figures")
summary = eda.response_summary()

eda.plot_likert_distribution(likert_cols, save=True)
eda.plot_correlation_heatmap(likert_cols, save=True)
eda.missing_data_report(save=True)

for col in demo_cols[:2]:
    eda.plot_demographic_breakdown(col, save=True)

plt.close("all")
print(f"✅ Generated {len(eda.figures)} figures")

### 2.4 Statistical Analysis

In [ ]:
stats = SurveyStats(clean_df)

# Reliability
alpha = stats.cronbachs_alpha(likert_cols)
print(f"✅ Cronbach's α = {alpha['alpha']} ({alpha['interpretation']})")

# Group comparison
comparison = stats.compare_groups("q1", "age_group")
print(f"✅ Group comparison: p = {comparison['p_value']}")

# Correlations
correlations = stats.correlation_matrix(likert_cols[:5])
print(f"✅ Correlation matrix: {correlations['correlation_matrix'].shape}")

# Chi-square
chi2 = stats.chi_square_test("age_group", "gender")
print(f"✅ Chi-square: χ² = {chi2['chi2']}, p = {chi2['p_value']}")

# Factor analysis
fa = stats.factor_analysis(likert_cols, n_factors=2)
print(f"✅ Factor analysis: {fa['n_factors']} factors, KMO = {fa['kmo']}")

### 2.5 ML Analysis

In [ ]:
# Classification
classifier = SurveyClassifier(clean_df)
classifier.prepare_data(feature_cols=likert_cols, target_col="satisfaction_group")
model_results = classifier.run_model_comparison(cv_folds=5)
print(f"✅ Best model: {classifier.best_model_name}")

# Segmentation
segmenter = SurveySegmentation(clean_df)
segmenter.prepare_data(likert_cols)
optimal_k = segmenter.find_optimal_k(k_range=range(2, 7))
cluster_profiles = segmenter.fit_clusters(n_clusters=optimal_k["optimal_k"])
print(f"✅ Segments: {optimal_k['optimal_k']} clusters")

plt.close("all")

---
## 3. Build the Report

Now we assemble all results into a structured HTML report.

In [ ]:
report = ReportGenerator(clean_df)

# Set metadata
report.set_metadata(
    title="Quarterly Survey Analysis Report — Q4 2024",
    author="Research Analytics Team",
    description=(
        "Comprehensive analysis of the Q4 2024 customer satisfaction survey, "
        f"covering {len(clean_df)} valid respondents across {len(likert_cols)} "
        "survey items."
    ),
)

print("✅ Report initialized")

### 3.1 Executive Summary

In [ ]:
# Build executive summary
n_original = len(df)
n_clean = len(clean_df)
n_removed = n_original - n_clean
best_score = model_results.iloc[0]["mean_score"]

exec_summary = f"""
<div style="background: #f0f9ff; padding: 20px; border-radius: 8px; border-left: 4px solid #2563eb;">
    <h3 style="color: #1e40af; margin-top: 0;">Key Findings</h3>
    <ul>
        <li><strong>Sample:</strong> {n_clean} valid respondents 
            ({n_removed} removed during quality screening from {n_original} total)</li>
        <li><strong>Scale Reliability:</strong> Cronbach's α = {alpha['alpha']} 
            ({alpha['interpretation']})</li>
        <li><strong>Satisfaction Distribution:</strong> 
            {(clean_df['satisfaction_group'] == 'Satisfied').mean():.1%} Satisfied, 
            {(clean_df['satisfaction_group'] == 'Neutral').mean():.1%} Neutral, 
            {(clean_df['satisfaction_group'] == 'Dissatisfied').mean():.1%} Dissatisfied</li>
        <li><strong>Group Differences:</strong> Age group comparison was 
            {'significant' if comparison['significant'] else 'not significant'} 
            (p = {comparison['p_value']})</li>
        <li><strong>Prediction Model:</strong> {classifier.best_model_name} achieved 
            {best_score:.1%} accuracy in predicting satisfaction</li>
        <li><strong>Respondent Segments:</strong> {optimal_k['optimal_k']} distinct 
            clusters identified</li>
    </ul>
</div>
"""

report.add_section("Executive Summary", exec_summary, section_type="text")
print("✅ Executive summary added")

### 3.2 Data Quality & Cleaning

In [ ]:
# Cleaning summary
cleaning_html = "<h3>Data Screening Steps</h3><ol>"
for step in cleaning_log:
    cleaning_html += f"<li><strong>{step['action']}:</strong> {step['details']}</li>"
cleaning_html += "</ol>"
cleaning_html += f"""
<p>
    <strong>Original sample:</strong> {n_original}<br>
    <strong>Final clean sample:</strong> {n_clean}<br>
    <strong>Retention rate:</strong> {n_clean/n_original:.1%}
</p>
"""

report.add_section("Data Quality & Cleaning", cleaning_html, section_type="text")

# Validation results
report.add_stats_result("Data Validation", validation)

print("✅ Data quality section added")

### 3.3 Response Summary

In [ ]:
report.add_dataframe(
    "Response Summary",
    summary,
    description="Descriptive statistics for all survey variables.",
    max_rows=30,
)

print("✅ Response summary added")

### 3.4 Visualizations

In [ ]:
from pathlib import Path

for fig_path in eda.figures:
    if Path(fig_path).exists():
        fig_name = Path(fig_path).stem.replace("_", " ").title()
        report.add_figure(
            fig_name,
            fig_path,
            description=f"Generated from {len(clean_df)} respondents.",
        )

print(f"✅ {len(eda.figures)} figures added")

### 3.5 Reliability Analysis

In [ ]:
report.add_stats_result("Reliability Analysis (Cronbach's Alpha)", alpha)

# Item diagnostics table
item_diag = pd.DataFrame({
    "Item": list(alpha["item_total_correlations"].keys()),
    "Item-Total r": list(alpha["item_total_correlations"].values()),
    "α if Deleted": list(alpha["alpha_if_deleted"].values()),
}).set_index("Item")

report.add_dataframe(
    "Item Diagnostics",
    item_diag,
    description=(
        f"Item-level diagnostics for the {alpha['n_items']}-item scale. "
        "Items with 'α if Deleted' higher than the overall alpha "
        f"({alpha['alpha']}) may be candidates for removal."
    ),
)

print("✅ Reliability section added")

### 3.6 Group Comparisons

In [ ]:
report.add_stats_result(
    "Group Comparison: Q1 by Age Group",
    comparison,
)

# Run comparisons for all items
all_comparisons = []
for col in likert_cols:
    result = stats.compare_groups(col, "age_group")
    all_comparisons.append({
        "Item": col,
        "Test": result["test"],
        "Statistic": result["statistic"],
        "p-value": result["p_value"],
        "Effect Size": result["effect_size"],
        "Significant": "✅" if result["significant"] else "❌",
    })

comp_df = pd.DataFrame(all_comparisons)
report.add_dataframe(
    "All Items — Age Group Comparisons",
    comp_df,
    description=(
        "Statistical comparisons of each survey item across age groups. "
        "Test automatically selected based on normality assumptions."
    ),
)

print("✅ Group comparison section added")

### 3.7 Correlation Analysis

In [ ]:
report.add_dataframe(
    "Correlation Matrix",
    correlations["correlation_matrix"].round(4),
    description=(
        f"{correlations['method'].title()} correlations for Likert items "
        f"(n = {correlations['n']}). "
        f"Found {len(correlations['significant_pairs'])} significant pairs."
    ),
)

# Significant pairs table
if correlations["significant_pairs"]:
    sig_pairs_df = pd.DataFrame(correlations["significant_pairs"])
    sig_pairs_df["Strength"] = sig_pairs_df["correlation"].abs().apply(
        lambda x: "Strong" if x >= 0.7 else "Moderate" if x >= 0.4 else "Weak"
    )
    report.add_dataframe(
        "Significant Correlations",
        sig_pairs_df,
        description="Pairs with p < 0.05, sorted by correlation strength.",
    )

print("✅ Correlation section added")

### 3.8 Chi-Square Test

In [ ]:
report.add_stats_result(
    "Chi-Square: Age Group × Gender",
    {k: v for k, v in chi2.items()
     if not isinstance(v, pd.DataFrame)},
)

report.add_dataframe(
    "Contingency Table (Observed)",
    chi2["contingency_table"],
    description="Observed frequencies for Age Group × Gender.",
)

report.add_dataframe(
    "Expected Frequencies",
    chi2["expected_frequencies"],
    description="Expected frequencies under null hypothesis of independence.",
)

print("✅ Chi-square section added")

### 3.9 Factor Analysis

In [ ]:
# Main FA results (non-DataFrame items)
fa_summary = {
    "Bartlett's χ²": fa["bartlett_chi_sq"],
    "Bartlett's p": fa["bartlett_p"],
    "Bartlett Significant": fa["bartlett_significant"],
    "KMO": fa["kmo"],
    "KMO Interpretation": fa["kmo_interpretation"],
    "KMO Adequate": fa["kmo_adequate"],
    "Number of Factors": fa["n_factors"],
    "Rotation": fa["rotation"],
    "Extraction Method": fa["method"],
    "Cumulative Variance": f"{fa['variance_explained']['cumulative'][-1]:.1%}",
    "N Observations": fa["n_observations"],
}
report.add_stats_result("Factor Analysis — Adequacy & Extraction", fa_summary)

# Loadings table
report.add_dataframe(
    "Factor Loadings",
    fa["loadings"].round(4),
    description=(
        f"{fa['rotation'].title()}-rotated factor loadings. "
        "Loadings > 0.4 indicate meaningful item-factor associations."
    ),
)

# Item assignments
assignments_df = pd.DataFrame({
    "Item": list(fa["item_assignments"].keys()),
    "Assigned Factor": list(fa["item_assignments"].values()),
    "Loading": [
        fa["loadings"].loc[item, factor]
        for item, factor in fa["item_assignments"].items()
    ],
    "Communality": [
        fa["communalities"][item]
        for item in fa["item_assignments"].keys()
    ],
}).round(4)

report.add_dataframe(
    "Item-Factor Assignments",
    assignments_df,
    description="Each item assigned to the factor with its highest loading.",
)

# Variance explained
var_df = pd.DataFrame({
    "Factor": [f"Factor_{i+1}" for i in range(fa["n_factors"])],
    "Eigenvalue": fa["variance_explained"]["eigenvalues"],
    "Variance Explained": [f"{v:.1%}" for v in fa["variance_explained"]["per_factor"]],
    "Cumulative": [f"{v:.1%}" for v in fa["variance_explained"]["cumulative"]],
})

report.add_dataframe(
    "Variance Explained",
    var_df,
    description="Proportion of total variance explained by each factor.",
)

print("✅ Factor analysis section added")

### 3.10 Classification Results

In [ ]:
report.add_dataframe(
    "Model Comparison — Satisfaction Prediction",
    model_results,
    description=(
        "Cross-validated performance of multiple classifiers predicting "
        "satisfaction group from survey responses. "
        f"Best model: {classifier.best_model_name}."
    ),
)

# Classification report
cls_report = classifier.get_classification_report()
cls_df = pd.DataFrame(cls_report["classification_report"]).T
report.add_dataframe(
    f"Classification Report — {cls_report['model']}",
    cls_df.round(4),
    description="Precision, recall, and F1-score per class.",
)

# Confusion matrix as table
cm_df = pd.DataFrame(
    cls_report["confusion_matrix"],
    index=[f"Actual: {c}" for c in cls_report["target_names"]],
    columns=[f"Predicted: {c}" for c in cls_report["target_names"]],
)
report.add_dataframe(
    "Confusion Matrix",
    cm_df,
    description="Rows = actual class, columns = predicted class.",
)

# Feature importance
try:
    importance = classifier.feature_importance()
    report.add_dataframe(
        "Feature Importance (SHAP)",
        importance,
        description=(
            "Features ranked by mean absolute SHAP value. "
            "Higher values indicate stronger influence on predictions."
        ),
    )
    print("✅ Feature importance added")
except ImportError:
    print("⚠️ SHAP not available — skipping feature importance")

print("✅ Classification section added")

### 3.11 Segmentation Results

In [ ]:
# Cluster overview
seg_summary = {
    "Number of Clusters": optimal_k["optimal_k"],
    "Silhouette Score": segmenter.results["silhouette"],
    "Best Silhouette": optimal_k["best_silhouette"],
    "Cluster Sizes": segmenter.results["cluster_sizes"],
}
report.add_stats_result("Respondent Segmentation — Overview", seg_summary)

# Cluster profiles
report.add_dataframe(
    "Cluster Profiles — Mean Scores",
    cluster_profiles.round(3),
    description=(
        "Average Likert-scale response per cluster. "
        "Higher values indicate more positive responses."
    ),
)

# Cluster standard deviations
if "cluster_stds" in segmenter.results:
    report.add_dataframe(
        "Cluster Profiles — Standard Deviations",
        segmenter.results["cluster_stds"].round(3),
        description="Within-cluster variability for each item.",
    )

# Demographic profiles
demo_profiles = segmenter.profile_clusters_by_demographics(demo_cols)
for demo_col, profile in demo_profiles.items():
    report.add_dataframe(
        f"Cluster × {demo_col.replace('_', ' ').title()}",
        (profile * 100).round(1),
        description=(
            f"Percentage distribution of {demo_col} within each cluster."
        ),
    )

# PCA visualization
viz_df = segmenter.visualize_clusters()
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    viz_df["PC1"], viz_df["PC2"],
    c=viz_df["cluster"], cmap="Set2",
    alpha=0.6, s=50, edgecolors="white", linewidth=0.5,
)
ax.set_xlabel(f"PC1 ({segmenter.results['pca_variance'][0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({segmenter.results['pca_variance'][1]:.1%} variance)")
ax.set_title("Respondent Segments (PCA Projection)")
plt.colorbar(scatter, label="Cluster", ax=ax)
plt.tight_layout()

pca_path = "../outputs/figures/report_cluster_pca.png"
plt.savefig(pca_path, dpi=150, bbox_inches="tight")
plt.close("all")

report.add_figure(
    "Cluster Visualization (PCA)",
    pca_path,
    description=(
        f"2D PCA projection of {optimal_k['optimal_k']} respondent clusters. "
        f"PC1 explains {segmenter.results['pca_variance'][0]:.1%} of variance, "
        f"PC2 explains {segmenter.results['pca_variance'][1]:.1%}."
    ),
)

print("✅ Segmentation section added")

### 3.12 Methodology Notes

In [ ]:
methodology = """
<h3>Data Collection</h3>
<p>
    Survey data was collected using a structured online questionnaire with 
    {n_items} Likert-scale items (1 = Strongly Disagree to 5 = Strongly Agree), 
    demographic questions, and an open-ended feedback field.
</p>

<h3>Data Quality</h3>
<ul>
    <li><strong>Speeders:</strong> Respondents completing in &lt; 60 seconds were removed</li>
    <li><strong>Straightliners:</strong> Respondents with &gt; 95% identical Likert responses were removed</li>
    <li><strong>Missing data:</strong> Handled via median imputation for numeric variables</li>
</ul>

<h3>Statistical Methods</h3>
<ul>
    <li><strong>Reliability:</strong> Cronbach's alpha with item-level diagnostics</li>
    <li><strong>Group comparisons:</strong> Auto-selected based on normality 
        (Shapiro-Wilk) and group count (t-test/Mann-Whitney for 2 groups, 
        ANOVA/Kruskal-Wallis for 3+)</li>
    <li><strong>Effect sizes:</strong> Cohen's d, eta-squared, Cramér's V as appropriate</li>
    <li><strong>Factor analysis:</strong> Maximum likelihood extraction with varimax rotation; 
        Kaiser criterion for factor retention</li>
    <li><strong>Chi-square:</strong> Test of independence with Cramér's V effect size</li>
</ul>

<h3>Machine Learning</h3>
<ul>
    <li><strong>Classification:</strong> Logistic Regression, Random Forest, 
        Gradient Boosting, and XGBoost compared via 5-fold stratified CV</li>
    <li><strong>Feature importance:</strong> SHAP (SHapley Additive exPlanations) values</li>
    <li><strong>Segmentation:</strong> K-Means clustering with silhouette analysis 
        for optimal k selection</li>
    <li><strong>Visualization:</strong> PCA projection for 2D cluster visualization</li>
</ul>

<h3>Software</h3>
<p>
    Analysis performed using the <strong>Survey ML Toolkit v0.1.0</strong> 
    (Python 3.10+), built on pandas, scikit-learn, scipy, statsmodels, 
    and SHAP.
</p>
""".format(n_items=len(likert_cols))

report.add_section("Methodology", methodology, section_type="text")
print("✅ Methodology section added")

---
## 4. Generate the Report

In [ ]:
print(f"\n📊 Report Summary:")
print(f"   Title: {report.title}")
print(f"   Author: {report.author}")
print(f"   Sections: {len(report.sections)}")
print(f"   Section titles:")
for i, section in enumerate(report.sections):
    print(f"     {i+1}. [{section['type']}] {section['title']}")

### 4.1 Generate HTML Report

In [ ]:
html_path = report.generate(
    output_path="../outputs/reports/q4_2024_survey_report.html",
    auto_sections=False,
)

print(f"\n✅ HTML Report generated!")
print(f"   📄 Path: {html_path}")
print(f"   📄 Size: {Path(html_path).stat().st_size / 1024:.1f} KB")

### 4.2 Generate PDF Report (Optional)

Requires `weasyprint`: `pip install survey-ml-toolkit[reporting]`

In [ ]:
try:
    pdf_path = report.generate_pdf(
        output_path="../outputs/reports/q4_2024_survey_report.pdf",
    )
    print(f"✅ PDF Report generated!")
    print(f"   📄 Path: {pdf_path}")
    print(f"   📄 Size: {Path(pdf_path).stat().st_size / 1024:.1f} KB")
except ImportError:
    print("⚠️ weasyprint not installed — skipping PDF generation")
    print("   Install with: pip install survey-ml-toolkit[reporting]")

---
## 5. Custom Report Builder

You can also build reports incrementally for different audiences.

### 5.1 Executive Summary Report (Minimal)

In [ ]:
exec_report = ReportGenerator(clean_df)
exec_report.set_metadata(
    title="Executive Summary — Q4 2024 Survey",
    author="Research Analytics Team",
    description="High-level findings for leadership review.",
)

# Only key findings
exec_report.add_section("Key Findings", exec_summary, section_type="text")

exec_report.add_dataframe(
    "Satisfaction Distribution",
    clean_df["satisfaction_group"].value_counts().to_frame("Count"),
    description="Overall satisfaction breakdown.",
)

exec_report.add_dataframe(
    "Top Performing Model",
    model_results.head(1),
    description="Best classification model for satisfaction prediction.",
)

exec_path = exec_report.generate(
    output_path="../outputs/reports/exec_summary.html",
    auto_sections=False,
)
print(f"✅ Executive report: {exec_path}")

### 5.2 Technical Appendix Report

In [ ]:
tech_report = ReportGenerator(clean_df)
tech_report.set_metadata(
    title="Technical Appendix — Q4 2024 Survey",
    author="Research Analytics Team",
    description="Detailed statistical outputs and diagnostics.",
)

# All statistical details
tech_report.add_stats_result("Cronbach's Alpha", alpha)
tech_report.add_dataframe("Item Diagnostics", item_diag)
tech_report.add_dataframe("Factor Loadings", fa["loadings"].round(4))
tech_report.add_dataframe("Variance Explained", var_df)
tech_report.add_dataframe(
    "Full Correlation Matrix",
    correlations["correlation_matrix"].round(4),
)
tech_report.add_dataframe("All Group Comparisons", comp_df)
tech_report.add_dataframe("Model Comparison", model_results)
tech_report.add_dataframe("Classification Metrics", cls_df.round(4))
tech_report.add_dataframe("Confusion Matrix", cm_df)
tech_report.add_dataframe("Cluster Profiles", cluster_profiles.round(3))
tech_report.add_section("Methodology", methodology, section_type="text")

tech_path = tech_report.generate(
    output_path="../outputs/reports/technical_appendix.html",
    auto_sections=False,
)
print(f"✅ Technical appendix: {tech_path}")

---
## 6. Summary

In [ ]:
from pathlib import Path

reports_dir = Path("../outputs/reports")
figures_dir = Path("../outputs/figures")

print("=" * 60)
print("REPORT GENERATION COMPLETE")
print("=" * 60)

print(f"\n📄 Reports Generated:")
if reports_dir.exists():
    for f in sorted(reports_dir.glob("*")):
        size = f.stat().st_size / 1024
        print(f"   • {f.name} ({size:.1f} KB)")

print(f"\n📊 Figures Generated:")
if figures_dir.exists():
    for f in sorted(figures_dir.glob("*.png")):
        size = f.stat().st_size / 1024
        print(f"   • {f.name} ({size:.1f} KB)")

print(f"\n📊 Analysis Summary:")
print(f"   Respondents: {len(clean_df)}")
print(f"   Survey items: {len(likert_cols)}")
print(f"   Statistical tests: {len(stats.get_all_results())}")
print(f"   ML models: {len(classifier.models)}")
print(f"   Segments: {optimal_k['optimal_k']}")
print(f"   Report sections: {len(report.sections)}")
print(f"\n✅ All done! Open the HTML reports in your browser to review.")